In [10]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from matplotlib.ticker import PercentFormatter

from fx_forecasting.config import load_experiment_config

split = "final_test"  # "validation" or "final_test"

# Jupyter starts in this notebook's folder, so the project root is one level up.
ROOT = Path("..")
periods = load_experiment_config(ROOT / "configs" / "experiment.yaml").annualization_periods
image_dir = ROOT / "images" / split
image_dir.mkdir(parents=True, exist_ok=True)

# cash is identical to random_walk, which never takes a position
is_shown = (pl.col("scenario") == "base_cost") & (pl.col("strategy") != "cash")
daily = pl.read_csv(ROOT / f"outputs/backtests/{split}_daily.csv", try_parse_dates=True).filter(is_shown)
strategy_metrics = pl.read_csv(ROOT / f"outputs/tables/{split}_strategy_metrics.csv").filter(is_shown)

strategies = sorted(daily["strategy"].unique(), key=lambda name: (name == "buy_and_hold", name))
colors = dict(zip(strategies, plt.get_cmap("tab10").colors, strict=False)) | {
    "buy_and_hold": "black"
}
plt.rcParams.update({"figure.figsize": (10, 5), "axes.grid": True, "grid.alpha": 0.3})


def save(fig, name):
    fig.savefig(image_dir / f"{name}.png", dpi=150, bbox_inches="tight")

In [11]:
summary = (
    strategy_metrics.filter(pl.col("strategy") != "random_walk")
    .select(
        pl.col("strategy").alias("model"),
        pl.col("net_sharpe_ratio").round(2).alias("net_sharpe"),
        (pl.col("net_cumulative_return") * 100).round(1).alias("net_cumulative_return_pct"),
        (pl.col("net_maximum_drawdown") * 100).round(1).alias("max_drawdown_pct"),
    )
    .sort("net_sharpe", descending=True)
)
summary

model,net_sharpe,net_cumulative_return_pct,max_drawdown_pct
str,f64,f64,f64
"""buy_and_hold""",0.98,20.7,-12.2
"""historical_mean""",0.98,20.7,-12.2
"""tsmixer_external""",0.51,9.7,-12.0
"""arima""",0.47,6.8,-11.1
"""tsmixer""",-0.42,-9.1,-22.6


In [ ]:
fig, ax = plt.subplots()
for strategy in strategies:
    rows = daily.filter(pl.col("strategy") == strategy)
    linestyle = "--" if strategy == "buy_and_hold" else "-"
    ax.plot(
        rows["realized_return_date"],
        rows["net_equity"],
        label=strategy,
        color=colors[strategy],
        linestyle=linestyle,
    )
ax.set(
    title=f"Net equity after costs, including carry ({split})", xlabel="Date", ylabel="Net equity"
)
ax.legend()
save(fig, "net_equity")

In [ ]:
fig, ax = plt.subplots()
for strategy in strategies:
    rows = daily.filter(pl.col("strategy") == strategy)
    linestyle = "--" if strategy == "buy_and_hold" else "-"
    ax.plot(
        rows["realized_return_date"],
        rows["net_drawdown"],
        label=strategy,
        color=colors[strategy],
        linestyle=linestyle,
    )
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set(title=f"Net drawdown ({split})", xlabel="Date", ylabel="Drawdown from peak")
ax.legend()
save(fig, "net_drawdown")

In [ ]:
breakdown = (
    daily.group_by("strategy")
    .agg(
        spot=(pl.col("effective_position") * pl.col("realized_return")).mean() * periods,
        carry=(pl.col("effective_position") * pl.col("realized_carry")).mean() * periods,
        costs=-pl.col("transaction_cost").mean() * periods,
    )
    .with_columns(net=pl.col("spot") + pl.col("carry") + pl.col("costs"))
    .sort("strategy")
)

x = np.arange(breakdown.height)
width = 0.2
fig, ax = plt.subplots()
for offset, component in enumerate(("spot", "carry", "costs", "net")):
    ax.bar(x + (offset - 1.5) * width, breakdown[component], width, label=component)
ax.set_xticks(x, breakdown["strategy"])
ax.axhline(0, color="black", linewidth=0.8)
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set(title=f"Average annual return by source ({split})", ylabel="Return per year")
ax.legend()
save(fig, "return_breakdown")

In [ ]:
years = strategy_metrics["observation_count"][0] / periods
# random_walk never trades, so its Sharpe ratio is undefined.
# Standard error from Lo (2002), assuming i.i.d. returns.
sharpe = (
    strategy_metrics.filter(pl.col("strategy") != "random_walk")
    .sort("strategy")
    .with_columns(standard_error=((1 + pl.col("net_sharpe_ratio") ** 2 / 2) / years).sqrt())
)

fig, ax = plt.subplots()
ax.bar(
    sharpe["strategy"],
    sharpe["net_sharpe_ratio"],
    yerr=sharpe["standard_error"],
    capsize=4,
    ecolor="tab:gray",
    color=[colors[strategy] for strategy in sharpe["strategy"]],
)
ax.axhline(0, color="black", linewidth=0.8)
ax.set(title=f"Net Sharpe ratio with ±1 standard error ({split})", ylabel="Annualized Sharpe ratio")
save(fig, "sharpe_ratio")

In [ ]:
position_mix = (
    daily.group_by("strategy")
    .agg(
        long=(pl.col("effective_position") == 1).mean(),
        flat=(pl.col("effective_position") == 0).mean(),
        short=(pl.col("effective_position") == -1).mean(),
    )
    .sort("strategy")
)

fig, ax = plt.subplots()
left = np.zeros(position_mix.height)
for side, color in (("long", "tab:green"), ("flat", "tab:gray"), ("short", "tab:red")):
    ax.barh(position_mix["strategy"], position_mix[side], left=left, label=side, color=color)
    left += position_mix[side].to_numpy()
ax.xaxis.set_major_formatter(PercentFormatter(1.0))
ax.set(title=f"Share of days long, flat and short ({split})", xlabel="Share of days")
ax.legend(loc="upper left", bbox_to_anchor=(1, 1))
save(fig, "position_mix")

In [ ]:
forecasts = pl.read_csv(ROOT / f"outputs/forecasts/{split}_forecasts.csv", try_parse_dates=True)
abs_errors = (
    forecasts.with_columns(abs_error=(pl.col("predicted_return") - pl.col("realized_return")).abs())
    .pivot(on="model", index="forecast_origin", values="abs_error")
    .sort("forecast_origin")
)
models = [model for model in strategies if model in abs_errors.columns and model != "random_walk"]
random_walk_error = abs_errors["random_walk"].to_numpy()
random_walk_mae = random_walk_error.mean()

# Diebold-Mariano test on absolute errors
rows = []
for model in models:
    difference = abs_errors[model].to_numpy() - random_walk_error
    standard_error = difference.std(ddof=1) / np.sqrt(len(difference))
    rows.append(
        {
            "model": model,
            "relative_mae": difference.mean() / random_walk_mae,
            "interval_95": 1.96 * standard_error / random_walk_mae,
            "dm_statistic": difference.mean() / standard_error,
        }
    )
relative_mae = pl.DataFrame(rows)

fig, ax = plt.subplots()
ax.bar(
    relative_mae["model"],
    relative_mae["relative_mae"],
    yerr=relative_mae["interval_95"],
    capsize=4,
    ecolor="tab:gray",
    color=[colors[model] for model in relative_mae["model"]],
)
ax.axhline(0, color="black", linewidth=0.8)
ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=1))
ax.set(
    title=f"Return MAE relative to the random walk, with 95% interval ({split})",
    ylabel="MAE vs random walk (above 0 = worse)",
)
save(fig, "mae_vs_random_walk")
relative_mae

In [ ]:
fig, ax = plt.subplots()
for model in models:
    saved_error = (abs_errors["random_walk"] - abs_errors[model]).cum_sum()
    ax.plot(abs_errors["forecast_origin"], saved_error, label=model, color=colors[model])
ax.axhline(0, color="black", linewidth=0.8)
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set(
    title=f"Cumulative absolute error versus the random walk ({split})",
    xlabel="Forecast origin",
    ylabel="Rising = beating the random walk",
)
ax.legend()
save(fig, "cumulative_error_vs_random_walk")